# Predicting Customer Churn in a Telecommunications Company
## A Business Intelligence & Data Mining Project

---

**Business Context:** Customer churn — when subscribers cancel their service — is one of the most critical challenges in telecommunications. Research consistently shows that retaining existing customers is significantly more cost-effective than acquiring new ones, making churn prediction a high-value business intelligence problem (Sikri et al., 2024). The model developed in this project is intended to support the Marketing and CRM teams in executing targeted retention campaigns by identifying high-risk customers, enabling the organisation to allocate resources efficiently and reduce revenue loss.

**Objective:** Apply the full BI and data mining lifecycle to build a predictive model that identifies at-risk customers and translates findings into actionable retention strategies.

**Dataset:** The publicly available IBM Telco Customer Churn dataset (IBM, 2018), containing 7,043 customer records with 21 features covering demographics, account information, and service subscriptions.

---
# Stage 1: Problem Definition & Data Scoping

## 1.1 Business Problem

TelcoPlus, a fictional telecommunications provider, faces an escalating churn rate that erodes its subscriber base. The VP of Customer Retention has commissioned a data-driven investigation to answer: *"Which customers are most likely to churn, and what are the key drivers behind their decision to leave?"*

This problem is strategically significant because customer retention is far more cost-effective than acquisition, with studies estimating acquisition costs at five to twenty-five times higher than retention costs (Sikri et al., 2024). In the telecommunications sector specifically, annual churn rates frequently exceed 30%, making it one of the most affected industries (Chang et al., 2024). A predictive churn model directly enables targeted offers, personalised outreach, and service improvements — shifting the retention strategy from reactive to proactive.

The intended users of this model are the Marketing and CRM teams, who would use individual customer churn risk scores to prioritise retention campaigns, allocate discount budgets, and design segment-specific interventions. The key decision the model supports is: *which customers should receive retention offers, and what type of offer would be most effective?*

## 1.2 Project Objectives

1. Explore the dataset to identify patterns and early indicators of churn.
2. Build and compare multiple classification models to predict customer churn.
3. Interpret model outputs to identify the most important churn drivers.
4. Apply customer segmentation to identify distinct behavioural profiles.
5. Recommend concrete, data-backed retention strategies with clear business justification.

## 1.3 Dataset Justification

The Telco Customer Churn dataset is well-suited because it represents a realistic scenario with a binary target variable (Churn: Yes/No), a mix of data types requiring meaningful pre-processing, and a class imbalance (~26–29% churn) that mirrors real-world conditions. It has been widely used in peer-reviewed churn prediction research as a benchmark dataset (Sikri et al., 2024; Sana et al., 2022; Chang et al., 2024).

In [1]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)
print(f"Churn distribution:\n{df['Churn_Binary'].value_counts(normalize=True).round(3)}")

NameError: name 'pd' is not defined

## 1.4 Data Loading & Initial Inspection

In [ ]:
# ─── Imports ───────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, f1_score, accuracy_score
)
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100
print("All libraries loaded successfully.")

In [ ]:
# ─── Load the IBM Telco Customer Churn dataset ────────────
# Source: IBM Sample Data Sets (IBM, 2018), available via Kaggle and GitHub.
# Hosted in the project repository for reproducibility.

url = 'https://raw.githubusercontent.com/K-H-SOE/DataMining/main/WA_Fn-UseC_-Telco-Customer-Churn.csv'

try:
    df = pd.read_csv(url)
    print("Dataset loaded from project GitHub repository.")
except Exception:
    # Fallback: load from local file if no internet access
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
    print("Dataset loaded from local CSV file.")

print(f"Dataset shape: {df.shape}")
print(f"Churn rate: {df['Churn'].value_counts(normalize=True)['Yes']:.1%}")
df.head()

### Dataset Description

The dataset is the **IBM Telco Customer Churn** dataset (IBM, 2018), a publicly available real-world dataset originally published by IBM as a sample data resource and widely hosted on Kaggle. It contains **7,043 customer records** and **21 features** covering three categories:

- **Demographics:** gender, SeniorCitizen, Partner, Dependents
- **Account information:** tenure, Contract, PaperlessBilling, PaymentMethod, MonthlyCharges, TotalCharges
- **Services subscribed:** PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies

The binary target variable `Churn` (Yes/No) indicates whether the customer left the provider in the last month. The dataset exhibits a class imbalance of approximately 73% retained vs 27% churned, which mirrors real-world churn conditions.

**Limitations of the dataset:**
- It represents a **static snapshot** — customer behaviour is captured at a single point in time, whereas real churn risk evolves dynamically.
- It **lacks behavioural variables** such as call quality metrics, customer service complaint history, network outage exposure, and competitor pricing data that could improve predictive accuracy.
- **No timestamps** are provided, so seasonal or temporal patterns in churn cannot be analysed.
- The dataset is relatively **small** (7,043 rows), which limits the complexity of models that can be trained without overfitting.

In [ ]:
# ─── Data overview ──────────────────────────────────────────
print("Data Types:\n"); print(df.dtypes)
print(f"\nMissing/blank values: TotalCharges = {df['TotalCharges'].isin([' ']).sum()}")

---
# Stage 2: Exploratory Data Analysis & Pre-processing

With the business problem defined, we now explore, clean, and prepare the data for analysis. This stage is critical for understanding the data's structure, identifying quality issues, and uncovering initial patterns that will guide model selection. Effective data visualisation is a key business intelligence skill for communicating complex data clearly to stakeholders (Chang et al., 2024).

## 2.1 Data Cleaning

The `TotalCharges` column contains 11 blank entries stored as whitespace strings. These correspond to customers with zero tenure (brand-new subscribers) and are imputed with 0. The target variable is encoded as a binary integer for modelling purposes.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)
df['Churn_Binary'] = (df['Churn'] == 'Yes').astype(int)
print(f"Churn distribution:\n{df['Churn_Binary'].value_counts(normalize=True).round(3)}")

## 2.2 Target Variable & Numeric Feature Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, col, title in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges'],
                           ['Tenure (months)', 'Monthly Charges (£)', 'Total Charges (£)']):
    for label, color in zip(['No', 'Yes'], ['#2ecc71', '#e74c3c']):
        ax.hist(df[df['Churn']==label][col], bins=30, alpha=0.5, label=f'Churn={label}', color=color, density=True)
    ax.set_title(title, fontsize=13, fontweight='bold'); ax.legend()
plt.tight_layout(); plt.show()

print(f"Median tenure — Churned: {df[df['Churn']=='Yes']['tenure'].median():.0f}, Retained: {df[df['Churn']=='No']['tenure'].median():.0f}")
print(f"Median monthly charge — Churned: £{df[df['Churn']=='Yes']['MonthlyCharges'].median():.2f}, Retained: £{df[df['Churn']=='No']['MonthlyCharges'].median():.2f}")

**Business Interpretation:** Churned customers have substantially lower tenure, suggesting the first 12–18 months represent a critical risk window before loyalty develops. From a business perspective, this means TelcoPlus should invest heavily in onboarding and early engagement to help new customers perceive value before they consider leaving. Churned customers also pay higher monthly charges, indicating price sensitivity is a key driver — customers facing higher costs may perceive lower value relative to what they are paying, making them more receptive to competitor offers (Chang et al., 2024). Total charges are lower for churned customers, which is partly driven by their shorter tenure — reinforcing that losing early-tenure customers forfeits significant future revenue.

## 2.3 Categorical Feature Analysis

In [ ]:
cat_cols = ['Contract', 'InternetService', 'OnlineSecurity', 'TechSupport',
            'PaymentMethod', 'SeniorCitizen', 'Partner']
fig, axes = plt.subplots(2, 4, figsize=(20, 10)); axes = axes.flatten()
for i, col in enumerate(cat_cols):
    ct = pd.crosstab(df[col], df['Churn'], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['#2ecc71', '#e74c3c'], edgecolor='black')
    axes[i].set_title(f'Churn by {col}', fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Proportion'); axes[i].legend(['No','Yes'], title='Churn')
    axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha='right')
fig.delaxes(axes[7]); plt.tight_layout(); plt.show()

**Business Interpretation:**

- **Contract type** shows the strongest relationship: month-to-month customers churn dramatically more, as the absence of a contractual commitment means zero switching cost. From a business perspective, incentivising longer-term contracts through discounts or loyalty rewards could introduce a financial and psychological barrier to switching.
- **Fiber optic** users churn more despite paying premium prices, suggesting a possible disconnect between the price premium and perceived service quality. TelcoPlus should investigate whether fiber subscribers experience service outages or slower-than-advertised speeds that drive dissatisfaction.
- Customers **without Online Security or Tech Support** churn significantly more. These services act as "stickiness" factors that increase perceived value and create dependency on the provider (Sikri et al., 2024). Offering free trials could increase engagement while reducing attrition.
- **Electronic check** payers show the highest churn — manual payment creates a recurring decision point where customers actively evaluate whether to continue paying. Automatic payment methods, by contrast, reduce this friction. Migrating customers to auto-pay through small incentives could improve retention.
- **Senior citizens** show elevated churn, which may reflect different service needs, lower digital literacy, or greater sensitivity to price or complexity. Tailored support and simplified plan options may help retain this demographic.

## 2.4 Feature Engineering & Pre-processing

In [ ]:
df_model = df.copy()
df_model.drop(columns=['customerID', 'Churn'], inplace=True)
binary_map = {'Yes': 1, 'No': 0, 'Male': 1, 'Female': 0}
for col in ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']:
    df_model[col] = df_model[col].map(binary_map)
df_model['Contract'] = df_model['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2})
df_model = pd.get_dummies(df_model, columns=['MultipleLines','InternetService','OnlineSecurity',
    'OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','PaymentMethod'], drop_first=True)
X = df_model.drop(columns=['Churn_Binary']); y = df_model['Churn_Binary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train[['tenure','MonthlyCharges','TotalCharges']] = scaler.fit_transform(X_train[['tenure','MonthlyCharges','TotalCharges']])
X_test[['tenure','MonthlyCharges','TotalCharges']] = scaler.transform(X_test[['tenure','MonthlyCharges','TotalCharges']])
print(f"Training: {X_train.shape[0]} | Test: {X_test.shape[0]} | Features: {X_train.shape[1]}")

---
# Stage 3: Data Mining & Business Insights

Based on the EDA findings — particularly the strong influence of contract type, tenure, and internet service on churn — we now build predictive models to quantify these relationships and generate actionable churn risk scores.

## 3.1 Model Selection Rationale

A critical decision in any data mining project is selecting appropriate algorithms. The choice must balance predictive performance with interpretability, as models that cannot be explained to business stakeholders will not be trusted or acted upon (Chang et al., 2024). Research in telecom churn prediction has employed a wide range of algorithms, with Logistic Regression, Decision Trees, Random Forests, and Gradient Boosting among the most commonly used (Sikri et al., 2024; Sana et al., 2022). We selected four algorithms offering different trade-offs:

- **Logistic Regression:** Baseline model offering high interpretability through direct coefficients, enabling clear identification of which features increase or decrease churn probability. The trade-off is that it assumes linear feature relationships, which may miss complex interactions.
- **Decision Tree:** Produces human-readable, rule-based outputs ideal for operational teams who need clear decision logic. However, single trees are prone to overfitting and may not generalise well.
- **Random Forest:** An ensemble of many decision trees that reduces overfitting and captures non-linear feature interactions. The trade-off is reduced interpretability compared to a single tree or logistic regression.
- **Gradient Boosting:** Sequentially builds trees that correct previous errors, typically achieving the highest accuracy on structured tabular data (Chang et al., 2024). The trade-off is increased complexity and lower interpretability.

By comparing all four, we assess whether the additional complexity of ensemble methods provides meaningful performance gains over simpler, more interpretable approaches.

## 3.2 Model Training & Cross-Validation

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.1, random_state=42)
}
cv_results = {}
print(f"{'Model':<25} {'CV Accuracy':>12} {'CV F1':>10} {'CV AUC':>10}")
print("-" * 60)
for name, model in models.items():
    acc = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    f1 = cross_val_score(model, X_train, y_train, cv=5, scoring='f1')
    auc = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    cv_results[name] = {'accuracy': acc.mean(), 'f1': f1.mean(), 'roc_auc': auc.mean()}
    print(f"{name:<25} {acc.mean():.4f}±{acc.std():.3f} {f1.mean():.4f}±{f1.std():.3f} {auc.mean():.4f}±{auc.std():.3f}")